# Sesiunile 11–13 — laborator

In [ ]:
import io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False})

---
# Date și baseline

Context: prezicem dacă un client va accepta oferta unei campanii bancare.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
with zipfile.ZipFile(io.BytesIO(__import__("urllib.request").request.urlopen(URL).read())) as outer:
    with zipfile.ZipFile(io.BytesIO(outer.read("bank.zip"))) as inner:
        df = pd.read_csv(inner.open("bank-full.csv"), sep=";")

print("Dimensiune:", df.shape)
print("Rata de raspuns pozitiv:", (df["y"] == "yes").mean().round(3))
df.head()

In [ ]:
target = (df["y"] == "yes").astype(int)
X = df.drop(columns="y")
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)

col_numerice = X.select_dtypes(include="number").columns.tolist()
col_categorice = [c for c in X.columns if c not in col_numerice]

def fa_preprocesare(numerice, categorice):
    return ColumnTransformer([
        ("num", StandardScaler(), numerice),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorice),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Coloane numerice:", col_numerice)
print("Coloane categorice:", col_categorice)

## Inspecția datelor
Observă distribuția țintei și semnificația valorilor speciale.

In [ ]:
# Completeaza inspectia datelor.
print(df["y"].value_counts(normalize=True).rename("proportie"))
print("pdays = -1:", (df["pdays"] == ___).mean().round(3))
print(df[["age", "balance", "duration", "campaign", "pdays"]].describe().round(1))

## Split-ul de antrenare și test
Verifică dacă proporția claselor se păstrează.

In [ ]:
# Ce argument din train_test_split pastreaza proportia claselor?
print("Clase train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Clase test: ", y_test.value_counts(normalize=True).round(3).to_dict())

## Baseline
Compară un model care nu învață nimic cu problema reală.

In [ ]:
baseline = DummyClassifier(strategy=___).fit(X_train, y_train)
pred = baseline.predict(X_train)
print("Accuracy:", round(accuracy_score(y_train, pred), 3))
print("Recall:", round(recall_score(y_train, pred), 3))

---
# Tuning și cross-validation

## Pipeline de bază
Leagă preprocesarea de un model într-un singur obiect.

In [ ]:
model_baza = Pipeline([
    ("prep", ___),
    ("model", ___),
])
model_baza.fit(X_train, y_train)
print("Pipeline pregatit:", list(model_baza.named_steps))

## Stabilitatea evaluării
Raportează media și deviația standard pe fold-uri.

In [ ]:
scoruri = cross_val_score(model_baza, X_train, y_train, cv=cv, scoring=___)
print("ROC-AUC pe fold-uri:", scoruri.round(3))
print(f"Medie: {scoruri.mean():.3f} +/- {scoruri.std():.3f}")

## Tuning pentru Random Forest
Caută hiperparametrii care maximizează ROC-AUC.

In [ ]:
rf_pipe = Pipeline([
    ("prep", fa_preprocesare(col_numerice, col_categorice)),
    ("model", RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)),
])
grila = {"model__max_depth": ___, "model__min_samples_leaf": ___}
search = GridSearchCV(rf_pipe, grila, cv=cv, scoring="roc_auc", n_jobs=-1)
search.fit(X_train, y_train)
print("Cel mai bun ROC-AUC:", round(search.best_score_, 3))
print("Parametri:", search.best_params_)

---
# PCA și t-SNE

## Componentele care păstrează 95% din variație
Calculează pragul de reducere dimensională.

In [ ]:
X_cifre, y_cifre = __import__("sklearn.datasets").datasets.load_digits(return_X_y=True)
pca_full = PCA().fit(StandardScaler().fit_transform(X_cifre))
varianta = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = ___
print("Componente pentru 95%:", n_95)

## PCA versus t-SNE
Compară o proiecție liniară cu una orientată spre vecini.

In [ ]:
sample = StandardScaler().fit_transform(X_cifre[:2000])
Z_pca = ___
Z_tsne = ___
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, Z, title in [(axes[0], Z_pca, "PCA"), (axes[1], Z_tsne, "t-SNE")]:
    ax.scatter(Z[:, 0], Z[:, 1], c=y_cifre[:2000], cmap="tab10", s=6, alpha=0.7)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## PCA în Pipeline
Evaluează PCA ca pas de preprocesare.

In [ ]:
prep_pca = Pipeline([
    ("prep", fa_preprocesare(col_numerice, col_categorice)),
    ("pca", ___),
    ("model", LogisticRegression(max_iter=1000)),
])
scor_pca = cross_val_score(prep_pca, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"ROC-AUC cu PCA: {scor_pca.mean():.3f} +/- {scor_pca.std():.3f}")

---
# Feature engineering și selecție

## Features de business
Construiește trei coloane care pot explica răspunsul la campanie.

In [ ]:
SERVICII = ["housing", "loan", "default"]

def adauga_features(d):
    d = d.copy()
    d["age_group"] = ___
    d["balance_per_campaign"] = ___
    d["was_contacted_before"] = ___
    return d

X_fe = adauga_features(X_train)
print(X_fe[["age_group", "balance_per_campaign", "was_contacted_before"]].head())

## Features noi în model
Compară modelul de bază cu modelul care folosește features noi.

In [ ]:
col_num_fe = ___
col_cat_fe = ___
model_fe = Pipeline([
    ("fe", FunctionTransformer(adauga_features)),
    ("prep", fa_preprocesare(col_num_fe, col_cat_fe)),
    ("model", LogisticRegression(max_iter=1000)),
])
for name, model in [("baza", model_baza), ("cu features", model_fe)]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
    print(name, round(scores.mean(), 3), "+/-", round(scores.std(), 3))

## Selecția de features
Pune selecția în Pipeline și compară rezultatul.

In [ ]:
selector_pipe = Pipeline([
    ("fe", FunctionTransformer(adauga_features)),
    ("prep", fa_preprocesare(col_num_fe, col_cat_fe)),
    ("select", ___),
    ("model", LogisticRegression(max_iter=1000)),
])
score_select = cross_val_score(selector_pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"ROC-AUC cu selectie: {score_select.mean():.3f} +/- {score_select.std():.3f}")

---
# Clase nebalansate și decizia de business

## Metrici pentru clasa pozitivă
Privește dincolo de accuracy: precision, recall, F1 și PR-AUC.

In [ ]:
def metrici(y_true, prob, prag=0.5):
    pred = (prob >= prag).astype(int)
    return {"accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
            "PR-AUC": average_precision_score(y_true, prob)}

prob = cross_val_predict(model_baza, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print(pd.Series(metrici(y_train, prob)).round(3))

## Pragul după cost
Modifică ipotezele de cost și alege pragul optim.

In [ ]:
cost_missed = ___
cost_offer = ___
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
costs = []
for threshold in thresholds:
    pred = prob >= threshold
    costs.append(((~pred) & (y_train.values == 1)).sum() * cost_missed + pred.sum() * cost_offer)
best_threshold = thresholds[np.argmin(costs)]
print("Prag optim:", best_threshold, "| cost:", min(costs))

## Cross-validation cu o clasă rară
Compară distribuția clasei pozitive în KFold și StratifiedKFold.

In [ ]:
idx_pos = np.where(y_train.values == 1)[0]
idx_neg = np.where(y_train.values == 0)[0]
rng = np.random.RandomState(RANDOM_STATE)
idx_rare = np.concatenate([idx_neg, rng.choice(idx_pos, int(len(idx_neg) * 0.05 / 0.95), replace=False)])
y_rare = y_train.iloc[idx_rare]
for name, splitter in [("KFold", KFold(5)), ("StratifiedKFold", cv)]:
    positives = [int(y_rare.iloc[val].sum()) for _, val in splitter.split(np.zeros(len(y_rare)), y_rare)]
    print(name, positives)

---
# Încheiere

Ținta, preprocesarea, evaluarea și pragul trebuie alese împreună cu întrebarea de business.